# 3. Webアプリ作成 — 統合講義ノートブック

## この講義の目的

**「Python（Streamlit）だけで、実際に動く検索エンジンWebアプリをゼロから作れるようになる」** ことがゴールです。
座学でWebアプリとStreamlitの基礎を固めたうえで、`PROJECT ZERO`（架空企業の社内ナレッジ検索エンジン開発ミッション）を通じて、検索アプリ **Tech0 Search** を **6週間かけて1つのアプリを育てながら** 作り上げます。

---

## 全体の構成（C7：5週間で育て、6週目に発表）

| 章 | タイトル | 内容 |
|----|---------|------|
| 3.1 | この講座の前提 | Webアプリとは／フレームワーク選定 → **別冊『事前学習』** |
| 3.2 | 基本実装 | Streamlit入門・基本コンポーネント → **別冊『事前学習』** |
| **3.3** | 応用実装 — PROJECT ZERO | 検索エンジン **Tech0 Search** を1本のアプリとして育てる |
| &nbsp;&nbsp;3.3.1 | **W1** ｜ v0.1 | Googleの仕組み／Streamlit検索UI／**簡易検索（部分一致）** ＋「本文まで検索したいが本文が無い→どう集める？」の課題提起 |
| &nbsp;&nbsp;3.3.2 | **W2** ｜ v0.2 | Webクローラー（自動登録）で本文まで **pages_w2.json に自動収集**（検索はW1の部分一致を流用） |
| &nbsp;&nbsp;3.3.3 | **W3** ｜ DB化＋全文検索 | JSON → SQLite へ移行し、**本文まで全文検索**（DB化のあとに全文検索） |
| &nbsp;&nbsp;3.3.4 | **W4** ｜ v1.0 | TF-IDF ランキング（宿題ここまで） |
| &nbsp;&nbsp;3.3.5 | **W5** | 役員会に向けた発表準備 |
| &nbsp;&nbsp;3.3.6 | **W6** | 最終発表 |

> 📘 **事前学習（別冊）**：Webアプリの前提と Streamlit の基礎は `事前学習_Webアプリとフレームワーク・Streamlit基礎.md` にまとめています。本編に入る前に、まずそちらで基礎を固めてください。

3.1・3.2で学んだStreamlitとPythonの基礎が、そのまま実戦で活きてきます。


---
# 3.3 応用実装 — PROJECT ZERO

ここからは応用実装として、実践プロジェクト **PROJECT ZERO** に取り組みます。
架空の東証プライム上場企業「株式会社テクゼロンBANK」のCDO（最高デジタル責任者）**黒崎** からの指令を受け、社内ナレッジ検索エンジン **Tech0 Search** を、**1つのアプリとして6週間かけて段階的に育て**ていきます。

```
3.3.1 W1 ｜ v0.1  Streamlit検索UI × 簡易検索（部分一致）
        ↓  「タイトルしか検索できないな。本文まで探せ。だが本文はどう集める？」
3.3.2 W2 ｜ v0.2  Webクローラー（自動で pages_w2.json に集める）
        ↓  「JSONに貯め続けるのは限界だ。ちゃんとした器（DB）に移せ」
3.3.3 W3 ｜ DB化＋全文検索  JSON→SQLite に移し、本文まで全文検索する
        ↓  「並び順が甘い。Googleのように賢く並べろ」
3.3.4 W4 ｜ v1.0  TF-IDF ランキング（宿題はここまで）
        ↓  「役員会で説明できる状態にしろ」
3.3.5 W5        発表準備（ROI・Q&A・デモ）
3.3.6 W6        最終発表
```

> 💡 **育てる開発**：毎週ゼロから作り直すのではなく、**先週のアプリ（app.py）に機能を足していく**のがこのプロジェクトの肝です。W1で作った画面は最後まで生き続けます。


---
## 3.3.3 W3 ｜ Tech0 Search — データをDBへ移す


# PROJECT ZERO — W3
# データの器を JSON から DB（SQLite）へ

## CDOからの指令書 — W3

```
╔════════════════════════════════════════════╗
║  W3 指令書                    PROJECT ZERO  ║
╠════════════════════════════════════════════╣
║                                            ║
║  クローラーで自動収集、よくやった。              ║
║                                            ║
║  だが pages_w2.json に延々と追記していく方式は  ║
║  限界だ。件数が増えれば壊れる。重複も防げん。     ║
║  ちゃんとした「器」= データベースに移せ。        ║
║                                            ║
║  今週のゴール：                              ║
║   ① pages_w2.json を SQLite（DB）に移す     ║
║   ② 本文まで検索できる「全文検索」を作る        ║
║   ③ 検索が DB を読むように差し替える           ║
║  ※ 賢い並び替え（ランキング）は来週やる。        ║
║    今週は「器の入れ替え」に集中しろ。           ║
║                                  — 黒崎    ║
╚════════════════════════════════════════════╝
```

### 〈今週の位置づけ〉

W2までは、集めたデータを `pages_w2.json`（1つのテキストファイル）に貯めていました。
**でもクローラーを回すほど中身は一気に膨らみ、読み書きは重く、壊れやすくなります。**
だから W3 では、**手遅れになる前に** それを **SQLite データベース（`tech0_search.db`）** に移します。

```
W2まで： crawler → pages_w2.json（ファイル） → search（JSONを読む）
W3から： crawler → DB（SQLite）          → search（DBを読む）
```

> **今週の流れ**：まず器を入れ替え（Step 1・2 で **DB化**）、次に **全文検索（`search_fulltext`）を作り**（Step 3）、最後に検索の**読み込み元を DB に差し替え**ます（Step 4）。
> `search_fulltext` は `pages`（ページの**リスト**）を受け取る作りなので、**そのリストが JSON 由来か DB 由来かを気にしません**。だから読み込み元だけ差し替えれば済みます。


---
# Step 0：全体像 — 今週いじるのは「貯め方」だけ（抽象でつかむ）

> ### 🎯 〈このStepのねらい〉
>
> W3 は **「器の入れ替え」** が主題です。集める（W2）も探す（W1）も変えません。変えるのは **どこに貯めるか** の1点だけ。
> それを先に頭に入れておかないと、SQL が出てきた瞬間に「今なにを作っているのか」を見失います。
>
> **このStepが終わったときにできていること**
> - 検索エンジンの3つの仕事のうち、W3で替えるのは **② 貯める** だと言える
> - 「保存先が変わっても検索ロジックは変えなくていい」理由を説明できる
> - 今週の4つのStepが「器を替える」と「本文まで探す」の2本立てだと整理できる
>
> **触るファイル**：なし（設計だけ。手を動かすのは Step 1 から）

コードに入る前に、いま自分が全体のどこを触るのかを俯瞰します。ここでは**固有名詞（ファイル名）はまだ出しません。**

### 0-1. 検索エンジンの3つの仕事のうち、今週替えるのは「② 貯める」

| 検索エンジンの仕事 | 今の状態 | 今週の変化 |
|--------------------|----------|-----------|
| ① 集める | W2 のクローラーで自動収集できた | （変えない） |
| ② **貯める** | JSON に追記（クローラーで **一気に肥大**・重い・壊れやすい） | **DB（SQLite）に移す** ← 今週の主役 |
| ③ 探す・並べる | W1 の部分一致だけ | **本文まで探す「全文検索」を足す** |

### 0-2. データの流れは「保存先が変わるだけ」

```
W2まで： 集める → [ JSONファイル ] → 読み込んで検索
W3から： 集める → [ DB（SQLite） ] → 読み込んで検索
```

変わるのは **「どこから読むか」** の1点だけ。検索する処理そのものは中身を変えません。

### 0-3. だから今週は2本立て

1. **器を替える**（JSON → DB）… `schema`（器の設計図）→ `database`（読み書き）
2. **本文まで探す**（全文検索）… DB に本文が入ったから、本文まで検索できる

最後にこの2つを **「検索の読み込み先を DB に差し替える」** ことでつなぎ、完成です。

### 今週のStep

- **Step 1**：器の設計図（`schema.sql`）
- **Step 2**：DBに読み書き（`database.py`）
- **Step 3**：全文検索を作る（本文まで探す）
- **Step 4**：検索の読み込み先を DB に差し替える


---

## ✅ Step 0 まとめ

**このStepでやったこと**

| やったこと | ひとことで言うと |
|------------|------------------|
| 3つの仕事のどこを触るか確認した | 替えるのは **② 貯める** だけ |
| データの流れを比べた | 変わるのは「どこから読むか」の1点 |
| 今週を2本立てに整理した | ① 器を替える（DB化）／② 本文まで探す（全文検索） |

**いまの成果物の状態**

| ファイル | 状態 |
|----------|------|
| `app.py` | 🟡 W2のまま（検索はJSONを読んでいる） |
| `crawler.py` | ✅ W2で完成済み（W3は変更なし） |
| `schema.sql` | ⬜ 未着手（Step 1 で作る） |
| `database.py` | ⬜ 未着手（Step 2 で作る） |
| `search_fulltext.py` | ⬜ 未着手（Step 3 で作る） |

**➡️ 次は Step 1**：まず器の設計図＝`schema.sql` から。DBは「形を決めてから入れる」道具です。


---
# Step 1：schema.sql を書く（今週は pages 中心）

> ### 🎯 〈このStepのねらい〉
>
> DB は「先に器の形を決めてから、データを入れる」道具です。その **器の設計図が `schema.sql`**。
> 今週は欲張らず、**`pages` テーブル1本** に集中して器を立ち上げます。
>
> **このStepが終わったときにできていること**
> - `CREATE TABLE` でテーブルの列と型を定義できる
> - `UNIQUE` 制約が **同じURLの重複登録を防ぐ** ことを説明できる
> - `keywords` テーブルは W4用、ログ2表は W6用 と、今週の守備範囲を線引きできる
>
> **触るファイル**：`schema.sql`（新規作成）

DB の設計図が `schema.sql` です。ただし今週は **欲張りません**。

- ✅ **今週使うのは `pages` テーブル**（ページ本体）。まずここにデータを移すことに集中する。
- 🔵 `keywords` テーブル（TF-IDFスコア保存用）は **W4のTF-IDF** で活用するので、定義だけ用意しておく。
- 🔵 `search_logs` / `click_logs`（検索・クリックのログ）は **W6の発展課題** で扱う。今週は作らない。

> 型（テーブル定義）は先に見せますが、**今週の主役は `pages` 一本**です。器を最小構成で立ち上げ、データが移せることを確認するのがゴールです。


In [1]:
# schema.sql の内容を Python 文字列として定義する（W3：最小構成）
# 今週は pages テーブルが主役。keywords は W4 のTF-IDF用に定義だけしておく。

schema_sql = """-- schema.sql
-- Tech0 Search データベース設計（W3：データ基盤の最小構成）

-- pagesテーブル（メイン）
-- UNIQUE 制約：同じ URL は重複して登録できない
CREATE TABLE IF NOT EXISTS pages (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    url         TEXT NOT NULL UNIQUE,
    title       TEXT NOT NULL,
    description TEXT,
    full_text   TEXT,
    author      TEXT,
    category    TEXT,
    keywords    TEXT,                    -- カンマ区切りで保存（例: "DX,営業店,業務効率"）
    word_count  INTEGER DEFAULT 0,
    crawled_at  DATETIME,
    created_at  DATETIME DEFAULT CURRENT_TIMESTAMP,
    updated_at  DATETIME DEFAULT CURRENT_TIMESTAMP
);

-- keywordsテーブル（TF-IDFスコア保存用。実際に使うのは W4）
CREATE TABLE IF NOT EXISTS keywords (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    page_id     INTEGER NOT NULL,
    keyword     TEXT NOT NULL,
    tf_score    REAL DEFAULT 0.0,
    tfidf_score REAL DEFAULT 0.0,
    FOREIGN KEY (page_id) REFERENCES pages(id) ON DELETE CASCADE
);

-- インデックス作成（検索を高速化する）
CREATE INDEX IF NOT EXISTS idx_keyword ON keywords(keyword);
CREATE INDEX IF NOT EXISTS idx_page_id ON keywords(page_id);

-- ※ search_logs / click_logs テーブルは W6（発展課題）で追加します。
"""

# schema.sql ファイルに書き出す
with open("schema.sql", "w", encoding="utf-8") as f:
    f.write(schema_sql)

print("schema.sql を作成しました（pages / keywords）")


schema.sql を作成しました（pages / keywords）


### 〈schema.sqlの補足〉

✅ `UNIQUE` 制約：同じ URL のページを2回 INSERT しようとするとエラーになる（重複防止）

✅ `FOREIGN KEY ... ON DELETE CASCADE`：ページを削除するとキーワードも自動削除

✅ `CREATE INDEX`：キーワードで検索するときに高速化される

🔵 `keywords` は **W4のTF-IDF** で使う。今週は定義だけ。

🔵 `search_logs` / `click_logs` は **W6の発展課題** で追加する。


---

## ✅ Step 1 まとめ

**このStepでやったこと**

| やったこと | ひとことで言うと |
|------------|------------------|
| `CREATE TABLE pages` を書いた | 器の「列と型」を先に決めた |
| `UNIQUE` を付けた | 同じURLは2度登録できない（重複防止） |
| `keywords` を定義だけした | 使うのは W4 のTF-IDF |
| `CREATE INDEX` を書いた | 検索を速くするための下ごしらえ |
| `schema.sql` に書き出した | SQLをPythonから切り離してファイル管理 |

**いまの成果物の状態**

| ファイル | 状態 |
|----------|------|
| `app.py` | 🟡 W2のまま |
| `crawler.py` | ✅ W2完成版を流用 |
| `schema.sql` | ✅ **完成**（pages / keywords の2テーブル） |
| `database.py` | ⬜ 未着手（次の Step 2 で着手） |
| `search_fulltext.py` | ⬜ 未着手 |

**➡️ 次は Step 2**：設計図はできました。次はそこに **Python から読み書きする窓口** を作ります。


---
# Step 2：`database.py` で DB に読み書きする

> ### 🎯 〈このStepのねらい〉
>
> 器（`schema.sql`）ができても、Python から読み書きできなければ使えません。
> **DBとの窓口を `database.py` 1ファイルに集約** して、アプリ側が SQL を書かなくて済む形にします。
>
> **このStepが終わったときにできていること**
> - `get_connection()` / `init_db()` で DB に接続・初期化できる
> - `insert_page()` の `INSERT OR REPLACE` が **再クロール時の上書き** を実現していると説明できる
> - `get_all_pages()` が返すのは **W2までと同じ「辞書のリスト」** だと確認できる
> - **W2で集めた `pages_w2.json` を、まるごとDBに移せる**（今週の主目的）
>
> **触るファイル**：`database.py`（新規作成）＋ `data/tech0_search.db`（自動生成）

### 〈2-1：database.py を作ろう〉

DB への接続・初期化・データの取得・保存を1つのファイルにまとめる。
これが「DB担当」の中心ファイル。


In [2]:
import sqlite3
from pathlib import Path
from datetime import datetime

# DB ファイルのパス（data/ サブフォルダに保存する）
DB_PATH = Path("data/tech0_search.db")

def get_connection():
    """
    DB への接続を取得する。

    row_factory を設定することで、行データを辞書のように扱える。
    data/ フォルダが存在しない場合は自動で作成する。
    """
    DB_PATH.parent.mkdir(exist_ok=True)   # data/ フォルダがなければ作る
    conn = sqlite3.connect(str(DB_PATH))
    conn.row_factory = sqlite3.Row        # 行データを辞書のように扱う
    return conn

def init_db():
    """
    schema.sql を読み込んで DB を初期化する。

    CREATE TABLE IF NOT EXISTS を使っているので、
    すでにテーブルが存在する場合は何もしない。
    """
    conn = get_connection()
    with open("schema.sql", "r", encoding="utf-8") as f:
        conn.executescript(f.read())    # SQL ファイルをまとめて実行する
    conn.commit()
    conn.close()
    print("DB を初期化しました（data/tech0_search.db）")

# 動作確認
init_db()

DB を初期化しました（data/tech0_search.db）


### 〈ここで確認してほしいこと〉

✅ `get_connection()` でDBへ接続している

✅ `init_db()` で`schema.sql`を読み込んで DB を初期化している

✅ `data/tech0_search.db` ファイルが生成されていることを確認しよう


### 〈2-2：INSERT OR REPLACE でページを登録しよう〉

`INSERT OR REPLACE` は「同じ URL のデータがあれば上書き、なければ新規追加」する。

**W2 では、同じURLを2回クロールすると 2件登録されてしまっていた。**
`pages_w2.json` にただ追記（`append`）していただけで、
「すでに同じURLが入っていないか」を確認していなかったからだ。

JSONのままで重複を防ごうとすると、**登録のたびに全件をループしてURLを突き合わせる**必要がある。
件数が増えるほど、この確認は重くなっていく。

DBなら、Step 1 で `url` 列に付けた **`UNIQUE` 制約** と `INSERT OR REPLACE` の組み合わせで、
同じことが **SQL 1文で、しかも件数が増えても速いまま** できる。

| | W2（JSON） | W3（DB） |
|---|---|---|
| 重複を防ぐには | 自分で全件ループして確認する | `UNIQUE` 制約に任せる |
| 同じURLを再クロールすると | 2件目が増えてしまう | **既存データを上書き**して最新にできる |
| 件数が増えたら | 全件ループなので遅くなる | インデックスが効くので速いまま |


In [3]:
def _keywords_to_text(keywords) -> str:
    """keywords（リスト or 文字列）を DB 保存用のカンマ区切り文字列にする"""
    if not keywords:
        return ""
    if isinstance(keywords, str):
        return keywords
    return ",".join(keywords)


def insert_page(page: dict) -> int:
    """
    ページ情報を DB に登録する。

    INSERT OR REPLACE：同じ URL のデータがあれば上書き、なければ新規追加する。
    これにより「同じページを再クロールしたときに最新データに更新できる」。

    Args:
        page: ページ情報の辞書（crawl_url() の返り値と同形式）

    Returns:
        登録された行の id
    """
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        INSERT OR REPLACE INTO pages
            (url, title, description, full_text, author, category, keywords, word_count, crawled_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        page["url"],
        page["title"],
        page.get("description", ""),
        page.get("full_text", ""),
        page.get("author", ""),
        page.get("category", ""),
        _keywords_to_text(page.get("keywords")),   # リスト → "DX,営業店" にして保存
        page.get("word_count", 0),
        page.get("crawled_at", datetime.now().isoformat()),
    ))

    page_id = cursor.lastrowid    # 登録された行の id を取得する
    conn.commit()
    conn.close()
    return page_id


def get_all_pages() -> list:
    """全ページを登録日時の新しい順で取得する。"""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM pages ORDER BY created_at DESC")
    rows = cursor.fetchall()
    conn.close()
    pages = []
    for row in rows:
        pg = dict(row)
        # 保存時にカンマ区切りにした keywords をリストに戻す
        pg["keywords"] = [k.strip() for k in (pg.get("keywords") or "").split(",") if k.strip()]
        pages.append(pg)
    return pages    # sqlite3.Row を辞書に変換して返す


# 動作確認：テストデータを登録する
test_pages = [
    {"url": "https://example.com/dx-report",
     "title": "DX戦略レポート2024",
     "description": "営業店のDX推進について解説した報告書",
     "full_text": "DX 推進 で 営業店 の 効率 が 向上 デジタル 変革 事例",
     "keywords": "DX,営業店,デジタル変革",
     "word_count": 150,
     "crawled_at": "2024-01-15T10:00:00"},
    {"url": "https://example.com/ai-guide",
     "title": "AI活用ガイド",
     "description": "業務効率化のためのAI導入ガイド",
     "full_text": "AI 活用 で 業務 効率 化 を 進める 機械 学習 ツール",
     "keywords": "AI,機械学習,業務効率化",
     "word_count": 120,
     "crawled_at": "2024-02-20T15:00:00"},
]

for p in test_pages:
    pid = insert_page(p)
    print(f"登録完了：id={pid}  {p['title']}")

# 登録されたデータを確認する
all_pages = get_all_pages()
print(f"\n現在の登録ページ数：{len(all_pages)} 件")


def log_search(query: str, results_count: int, user_id: str = None) -> int:
    """
    検索ログを記録する（W6 の発展課題で実装予定）
    現在はスタブ（空の関数）として定義しています。
    W6 の発展課題で実際の記録処理を実装します。
    """
    pass  # W6 の発展課題で実装します

登録完了：id=1  DX戦略レポート2024
登録完了：id=2  AI活用ガイド

現在の登録ページ数：2 件


### 〈ここで確認してほしいこと〉

✅ `INSERT OR REPLACE` は「重複チェック不要」で簡潔に書ける（しかし注意点もあり、下部参照）

✅ `insert_page(page: dict) -> int:` でページ情報をDBに登録している

✅ `get_all_pages() -> list:` で全ページを登録日時の新しい順で取得している

**`INSERT` と `INSERT OR REPLACE` の違い：**

| コマンド | 同じ URL が既存の場合 |
|----------|---------------------|
| `INSERT` | エラーになる（UNIQUE 制約違反） |
| `INSERT OR REPLACE` | 既存のデータを削除して新規追加 |

>
> ⚠️注意｜ **このセルを複数回実行すると id が増えます**
>
> `INSERT OR REPLACE` は「削除してから再挿入」するため、実行のたびに id が増加します。
> `data/tech0_search.db` はノートブックと同じフォルダに**ファイルとして残り続ける**ため、
> リセットしたい場合は次の方法を使ってください：
> ```python
> import os
> os.remove("data/tech0_search.db")  # DB ファイルを削除して最初からやり直す
> ```


### 〈2-3：W2で集めたデータを、まるごとDBに移す〉

ここまでで、DBに **テストデータ2件** を入れました。
でも、あなたが本当に移したいのは **W2でクローラーに集めさせたデータ** のはずです。

```
いまの状態
  pages_w2.json  ← W2で集めたデータ（本文つき）  ……ここに取り残されている
  DB             ← テストデータ2件だけ
```

**これが今週の本題「器の入れ替え」です。** 中身を新しい器に引っ越しましょう。

やることは単純で、**JSONを読んで、1件ずつ `insert_page()` に渡すだけ** です。
`insert_page()` は Step 2-2 で作ったものをそのまま使います。新しく覚えることはありません。

```
pages_w2.json を json.load() で読む   ← W2 と同じ読み方
        ↓
1件ずつ insert_page() に渡す          ← Step 2-2 で作った関数
        ↓
DB に入る
```


In [4]:
import json   # JSONファイルを読むために使う（W2 と同じ）


def migrate_from_json(json_path: str = "pages_w2.json") -> int:
    """
    W2 で使っていた JSON のデータを、まるごと DB に移す。

    Args:
        json_path: 移行元のJSONファイル

    Returns:
        移行した件数
    """
    # ① JSONファイルを読む（W2 でやったのと同じ）
    with open(json_path, "r", encoding="utf-8") as f:
        pages = json.load(f)

    # ② 1件ずつ DB に入れる（Step 2-2 で作った insert_page を使うだけ）
    for p in pages:
        insert_page(p)

    return len(pages)


# ── 実行：W2 のデータを DB に移す ──
n = migrate_from_json("pages_w2.json")
print(f"✅ {n} 件を DB に移行しました")

# ── 移行後の DB の中身を確認する ──
pages = get_all_pages()
print(f"現在の DB の登録件数：{len(pages)} 件（テストデータ2件 ＋ 移行分）")
print()
for p in pages[:5]:
    honbun = "本文あり" if p.get("full_text") else "本文なし"
    print(f"  id={p['id']:>2}  [{honbun}]  {p['title'][:30]}")

✅ 34 件を DB に移行しました
現在の DB の登録件数：36 件（テストデータ2件 ＋ 移行分）

  id= 3  [本文なし]  営業店DX推進レポート
  id= 4  [本文なし]  住宅ローン審査AI導入の検討
  id= 5  [本文なし]  法人融資 稟議書の書き方ガイド
  id= 6  [本文なし]  AIチャットボットによる照会対応DX
  id= 7  [本文なし]  全店DX推進ロードマップ2026


### 〈ここで確認してほしいこと〉

✅ `migrate_from_json()` の中身は **`json.load()` ＋ `insert_page()` のループだけ**

✅ 移行後は、W1由来のデータ（**本文なし**）と、W2のクローラーが集めたデータ（**本文あり**）が
   1つのDBに混在している

> 💡 **本文の有無が、このあとの検索で効いてきます。**
> W1由来のデータは `title` と `description` しか持っていません。
> W2のクローラーが集めたデータだけが `full_text`（本文）を持っています。
> Step 3 の全文検索は、この **本文まで** 探しにいきます。

### 〈2-4：同じURLを2回入れたらどうなる？〉

Step 2-2 で `INSERT OR REPLACE` を使ったのは、**同じURLの重複を防ぐため** でした。
本当にそうなるか、**さっきの移行をもう一度実行して** 確かめてみましょう。

JSONに追記していたW2なら、同じ内容がもう1セット増えてしまうはずです。DBではどうなるでしょうか。


In [5]:
# 移行前の件数を記録しておく
before = len(get_all_pages())

# まったく同じJSONを、もう一度移行してみる
migrate_from_json("pages_w2.json")

after = len(get_all_pages())

print(f"1回目の移行後: {before} 件")
print(f"2回目の移行後: {after} 件")
print()

if before == after:
    print("✅ 件数が増えていない ＝ 同じURLは新規追加ではなく【上書き】された")
    print("   W2（JSONにappend）なら、同じデータが2セットに増えていました。")
else:
    print(f"❌ {after - before} 件増えてしまった（UNIQUE制約を確認しよう）")

1回目の移行後: 36 件
2回目の移行後: 36 件

✅ 件数が増えていない ＝ 同じURLは新規追加ではなく【上書き】された
   W2（JSONにappend）なら、同じデータが2セットに増えていました。


---

## ✅ Step 2 まとめ

**このStepでやったこと**

| やったこと | ひとことで言うと |
|------------|------------------|
| `get_connection()` を書いた | DBへの接続（`row_factory` で行を辞書のように扱う） |
| `init_db()` を書いた | `schema.sql` を流し込んでDBを初期化 |
| `insert_page()` を書いた | `INSERT OR REPLACE` ＝ あれば上書き・なければ追加 |
| `get_all_pages()` を書いた | DB → **辞書のリスト**（W2までと同じ形で返す） |
| テストデータを登録した | `data/tech0_search.db` が実際にできた |
| **`migrate_from_json()` を書いた** | **`pages_w2.json` を まるごとDBに移した（今週の本題）** |
| 同じJSONを2回移行してみた | 件数が増えない＝`UNIQUE` ＋ `INSERT OR REPLACE` が効いている |

**いまの成果物の状態**

| ファイル | 状態 |
|----------|------|
| `app.py` | 🟡 W2のまま（まだDBを見ていない） |
| `crawler.py` | ✅ W2完成版を流用 |
| `schema.sql` | ✅ 完成 |
| `database.py` | ✅ **完成**（init / insert / get の3点セット） |
| `data/tech0_search.db` | ✅ **W2のデータを移行済み**（本文つきデータも入った） |
| `search_fulltext.py` | ⬜ 未着手（次の Step 3 で着手） |

**➡️ 次は Step 3**：器の入れ替えは完了（データも引っ越し済み）。DBに入った **本文（`full_text`）まで探す** 全文検索を作ります。


### CDO黒崎からの途中レビュー

```
┌──────────────────────────────────────────────────┐
│  💬 CDO黒崎からの指令                               │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│                                                  │
│  クローラーで本文も取れるようになったな。               │
│  じゃあ、本文も検索できるようにしろ。                  │
│                                                  │
│  検索結果にはマッチした数を表示しろ。                  │
│  どれくらい関連性があるか、一目でわかるようにな。        │
│                                                  │
└──────────────────────────────────────────────────┘

```


---
# Step 3：全文検索を作る（本文まで検索する）

> ### 🎯 〈このStepのねらい〉
>
> DB に本文（`full_text`）が入りました。ここでようやく **本文まで探す＝全文検索** が作れます。
> いきなり関数を書かず、**1ページ → 全ページ → 並べ替え → preview** と段階を踏んで組み上げます。
>
> **このStepが終わったときにできていること**
> - `title + description + full_text + keywords` を1本の検索対象テキストにまとめられる
> - `count()` で出現回数（`match_count`）を数え、`sorted()` で降順に並べられる
> - `search_fulltext()` と補助関数 `_make_preview()` の役割分担を説明できる
>
> **触るファイル**：`search_fulltext.py`
> （Jupyter で関数を理解 → Step 3 の最後に `search_fulltext.py` へ書き出します）

### 全文検索とは？

W1 の検索は、主に `title` / `description` / `keywords` を見る **部分一致** でした。  
W2 のクローラーが、各ページの **本文（`full_text`）** まで集めてくれて、いま DB に貯まっています。

そこで W3 では、検索対象を

```
title + description + full_text + keywords
```

に広げます。このように **本文まで含めて検索する** ことを **全文検索** と呼びます。

### このステップの目的

0. **まず DB から検索対象データを読み込む**（`get_all_pages()`）
1. 検索対象の文字列を作る  
2. キーワードが何回出てきたかを数える（`match_count`）  
3. 検索結果を並べ替える  
4. preview（抜粋）を作る  
5. `search_fulltext.py` の形にまとめる

Step 3 が終わるころには **`search_fulltext.py` の中身が自然にできあがる** 状態を目指します。  
（※ 賢い並び替え＝TF-IDF は来週 W4。今週は `match_count` 版でOK）


## 3-0：まず、DBに移したデータを読み込む

W3 では検索対象を **DB（SQLite）** に移しました。だから全文検索も、**DB から読み込んだデータ** に対して行います。
`get_all_pages()`（Step 2 で作成）で全ページを取り出し、変数 `pages` に入れます。以降の 3-1〜3-9 は、この **DBのデータ** を検索します。

In [6]:
# DB からデータを読み込む（Step 2 で作った get_all_pages を使う）
pages = get_all_pages()
print(f"DBから読み込んだページ数：{len(pages)} 件")
if pages:
    print("先頭ページ:", pages[0]["title"])

DBから読み込んだページ数：36 件
先頭ページ: 営業店DX推進レポート


## 3-1：まずは検索対象の文字列を作ってみよう

検索をするには、まず  
**「どこを検索対象にするか」** を決める必要があります。

今回は、1ページ分のデータから

- title
- description
- full_text
- keywords

をつなげて、1つの検索対象文字列を作ります。

これによって、  
タイトルにも本文にもキーワードにもマッチできるようになります。


In [7]:
# まずは1ページ分のデータを使って、検索対象文字列を作ってみる
page = pages[0]

# title, description, full_text, keywords をまとめて1つの文字列にする
search_text = " ".join([
    page.get("title", ""),
    page.get("description", ""),
    page.get("full_text", ""),
    " ".join(page.get("keywords", [])),
])

print(search_text[:500])

営業店DX推進レポート 窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録  DX 営業店 業務効率


### 〈ここで確認してほしいこと〉

この `search_text` は、  
**そのページの「検索対象全部入りテキスト」** です。

つまり検索では、毎回

```
この検索対象テキストの中に
キーワードがあるか
```

を調べることになります。

この考え方が、全文検索の基本です。


## 3-2：キーワードが含まれているか調べてみよう

次に、ユーザーが入力した検索キーワードが  
このページに含まれているかを調べます。

まずはシンプルに、  
**含まれているかどうか** を確認してみましょう。


In [8]:
# 検索キーワード
query = "DX"

# 大文字小文字の違いを吸収するために lower() にして比較する
found = query.lower() in search_text.lower()

print("検索キーワード:", query)
print("含まれているか:", found)

検索キーワード: DX
含まれているか: True


## 3-3：何回出てくるか数えてみよう

検索結果を並べるためには、

```
ある / ない
```

だけでなく、

```
何回出てきたか
```

も重要です。

今回はシンプルに、  
`count()` を使ってキーワードの出現回数を数えます。


In [9]:
# 検索キーワードの出現回数を数える
count = search_text.lower().count(query.lower())

print("検索キーワード:", query)
print("出現回数:", count)

検索キーワード: DX
出現回数: 3


### 〈ここで覚えてほしいこと〉

この出現回数を  
**match_count** として扱います。

つまり、

- match_count が多いページほど
- より関連が高いとみなして
- 上に表示する

という考え方です。

これはまだ単純な方法ですが、  
検索システムの最初の一歩としては十分です。


## 3-4：全ページに対して検索してみよう

今までは1ページだけ見ていました。

でも実際の検索では、  
**保存されている全ページを順番に確認する** 必要があります。

ここで使うのが

```python
for page in pages:
```

です。

このコードでは、各ページごとに

1. 検索対象文字列を作る  
2. キーワードの出現回数を数える  
3. ヒットしたページだけ結果に追加する  

という処理を行います。


In [10]:
query = "DX"
results = []

# すべてのページを1件ずつ確認する
for page in pages:
    
    # そのページの検索対象文字列を作る
    text = " ".join([
        page.get("title", ""),
        page.get("description", ""),
        page.get("full_text", ""),
        " ".join(page.get("keywords", [])),
    ]).lower()

    # キーワードの出現回数を数える
    count = text.count(query.lower())

    # 1回以上出てきたページだけ結果に追加
    if count > 0:
        result = page.copy()              # 元データをコピー
        result["match_count"] = count     # 検索結果用に match_count を追加
        results.append(result)

print("検索結果件数:", len(results))
results[:2]

検索結果件数: 7


[{'id': 37,
  'url': 'https://kb.techzeron-bank.example/dx-report',
  'title': '営業店DX推進レポート',
  'description': '窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録',
  'full_text': '',
  'author': '田中太郎',
  'category': '事例',
  'keywords': ['DX', '営業店', '業務効率'],
  'word_count': 0,
  'crawled_at': '2026-09-23T16:21:17.193334',
  'created_at': '2026-09-23 07:21:17',
  'updated_at': '2026-09-23 07:21:17',
  'match_count': 3},
 {'id': 38,
  'url': 'https://kb.techzeron-bank.example/ai-screening',
  'title': '住宅ローン審査AI導入の検討',
  'description': 'AI事前審査で融資業務をDX化する企画提案',
  'full_text': '',
  'author': '鈴木花子',
  'category': '企画',
  'keywords': ['AI', '融資', '審査'],
  'word_count': 0,
  'crawled_at': '2026-09-23T16:21:17.195420',
  'created_at': '2026-09-23 07:21:17',
  'updated_at': '2026-09-23 07:21:17',
  'match_count': 1}]

## 3-5：検索結果を並べ替えてみよう

今の `results` には、ヒットしたページが入っています。

次にやりたいのは、  
**match_count の多い順に並べること** です。

このコードでは、`sorted()` を使って  
検索結果を並べ替えます。


In [11]:
# match_count の大きい順に並べる
sorted_results = sorted(
    results,
    key=lambda x: x["match_count"],
    reverse=True
)

print("並べ替え後の上位3件:")
for r in sorted_results[:3]:
    print(r["title"], "->", r["match_count"])

並べ替え後の上位3件:
DX戦略レポート2024 -> 4
営業店DX推進レポート -> 3
AIチャットボットによる照会対応DX -> 3


### 〈ここで確認してほしいこと〉

この時点で、すでに

```
検索
↓
ヒットしたページだけ残す
↓
match_count順に並べる
```

という検索システムの基本ができています。

ただし、検索結果としてはまだ少し味気ないです。

そこで次は、  
**preview（本文の一部）** を作って、検索結果っぽくします。


## 3-6：preview を作ってみよう

検索結果では、タイトルだけでなく  
**本文の一部が表示されること** が多いです。

これを preview と呼びます。

今回は、キーワードが見つかった場所の前後を切り出して、  
previewを作ります。


In [12]:
def make_preview(text: str, query: str, ctx: int = 80) -> str:
    """
    キーワードが見つかった位置の前後を切り出して preview を作る

    Args:
        text: 本文
        query: 検索キーワード
        ctx: 前後に何文字表示するか

    Returns:
        preview文字列
    """
    # キーワードの位置を探す
    pos = text.lower().find(query.lower())

    # 見つからない場合は先頭200文字を返す
    if pos == -1:
        return text[:200]

    # 前後の範囲を決める
    start = max(0, pos - ctx)
    end = min(len(text), pos + len(query) + ctx)

    # 前後を切り出して返す
    return "..." + text[start:end] + "..."

## 3-7：preview を試してみよう

今作った `make_preview()` を使って、  
検索キーワード周辺の本文を表示してみましょう。


In [13]:
query = "DX"
results = []

for page in pages:
    
    # 検索対象文字列を作る
    text = " ".join([
        page.get("title", ""),
        page.get("description", ""),
        page.get("full_text", ""),
        " ".join(page.get("keywords", [])),
    ]).lower()

    # 出現回数を数える
    count = text.count(query.lower())

    # ヒットしたページだけ結果に追加
    if count > 0:
        result = page.copy()
        result["match_count"] = count
        
        # previewは full_text があればそれを使い、なければ description を使う
        result["preview"] = make_preview(
            page.get("full_text") or page.get("description", ""),
            query
        )
        
        results.append(result)

# match_count順に並べる
results = sorted(results, key=lambda x: x["match_count"], reverse=True)

print("検索結果件数:", len(results))
results[:2]

検索結果件数: 7


[{'id': 1,
  'url': 'https://example.com/dx-report',
  'title': 'DX戦略レポート2024',
  'description': '営業店のDX推進について解説した報告書',
  'full_text': 'DX 推進 で 営業店 の 効率 が 向上 デジタル 変革 事例',
  'author': '',
  'category': '',
  'keywords': ['DX', '営業店', 'デジタル変革'],
  'word_count': 150,
  'crawled_at': '2024-01-15T10:00:00',
  'created_at': '2026-09-23 07:17:02',
  'updated_at': '2026-09-23 07:17:02',
  'match_count': 4,
  'preview': '...DX 推進 で 営業店 の 効率 が 向上 デジタル 変革 事例...'},
 {'id': 37,
  'url': 'https://kb.techzeron-bank.example/dx-report',
  'title': '営業店DX推進レポート',
  'description': '窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録',
  'full_text': '',
  'author': '田中太郎',
  'category': '事例',
  'keywords': ['DX', '営業店', '業務効率'],
  'word_count': 0,
  'crawled_at': '2026-09-23T16:21:17.193334',
  'created_at': '2026-09-23 07:21:17',
  'updated_at': '2026-09-23 07:21:17',
  'match_count': 3,
  'preview': '...窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録...'}]

## 3-8：検索ロジックを関数にまとめよう

3-7 までで、全文検索のロジックはすでに完成しています。

やっていることは次の通りでした。

```
検索対象文字列を作る
↓
キーワード出現回数を数える
↓
ヒットしたページを results に追加
↓
preview を付ける
↓
match_count順に並べる
```

ただし、このままだと  
毎回同じコードを書かなければいけません。

そこで、この処理を  
**`search_fulltext()` という関数**にまとめます。

また、preview を作る処理は少し長いので  
**補助関数 `_make_preview()`** として切り出します。

補助関数とは、  
**他の関数の中で使うサポート用の小さな関数**です。

こうすることで、コードが整理され、  
あとから `app.py` などから簡単に呼び出せるようになります。


In [14]:
def _make_preview(text: str, query: str, ctx: int = 80) -> str:
    """
    マッチ箇所周辺のプレビューを生成する。

    Args:
        text: 本文
        query: 検索キーワード
        ctx: 前後に表示する文字数

    Returns:
        preview文字列
    """

    # text または query が空なら空文字を返す
    if not text or not query:
        return ""

    # キーワードの位置を探す
    pos = text.lower().find(query.lower())

    # 見つからない場合は先頭200文字を返す
    if pos == -1:
        return (text[:200] + "...") if len(text) > 200 else text

    # 前後の切り出し範囲
    start = max(0, pos - ctx)
    end = min(len(text), pos + len(query) + ctx)

    preview = ""

    if start > 0:
        preview += "..."

    preview += text[start:end]

    if end < len(text):
        preview += "..."

    return preview


def search_fulltext(query: str, pages: list) -> list:
    """
    全文検索（本文含む）を実行し、マッチ数でスコアリングする。

    Args:
        query: 検索キーワード
        pages: ページリスト

    Returns:
        マッチしたページのリスト（スコア降順）
    """

    # 空検索なら空リストを返す
    if not query.strip():
        return []

    results = []

    # 検索キーワードを小文字化
    q = query.lower()

    # 全ページを順番に確認
    for page in pages:

        # 検索対象テキストを作る
        text = " ".join([
            page.get("title", ""),
            page.get("description", ""),
            page.get("full_text", ""),
            " ".join(page.get("keywords", [])),
        ]).lower()

        # 出現回数を数える
        count = text.count(q)

        # ヒットしたページだけ追加
        if count > 0:

            r = page.copy()

            # match_count を追加
            r["match_count"] = count

            # preview を追加
            r["preview"] = _make_preview(
                page.get("full_text") or page.get("description", ""),
                query
            )

            results.append(r)

    # match_count順に並べる
    results.sort(key=lambda x: x["match_count"], reverse=True)

    return results

## 3-9：search_fulltext() を試してみよう

3-8 で `search_fulltext()` が完成しました。

次は、この関数を実際に使って  
**検索が正しく動くか確認してみます。**

このコードでは次のことを行います。

1. 検索キーワードを指定する  
2. `search_fulltext()` を実行する  
3. 検索結果の件数を確認する  
4. 上位の検索結果を表示する  

検索エンジンがどのように結果を返しているかを  
イメージしながら実行してみてください。


In [15]:
# 検索キーワード
query = "DX"

# search_fulltext を実行
results = search_fulltext(query, pages)

# 検索結果件数を表示
print("検索結果件数:", len(results))

# 上位5件を表示
for r in results[:5]:
    print("----")
    # タイトル
    print("タイトル:", r["title"])
    # キーワード出現回数
    print("match_count:", r["match_count"])
    # preview（本文の一部）
    print("preview:", r["preview"][:120])

検索結果件数: 7
----
タイトル: DX戦略レポート2024
match_count: 4
preview: DX 推進 で 営業店 の 効率 が 向上 デジタル 変革 事例
----
タイトル: 営業店DX推進レポート
match_count: 3
preview: 窓口業務のペーパーレス化で事務時間を30%削減したDXプロジェクトの記録
----
タイトル: AIチャットボットによる照会対応DX
match_count: 3
preview: コールセンターにAIチャットボットを導入し照会対応を効率化したDX事例
----
タイトル: 全店DX推進ロードマップ2026
match_count: 3
preview: 勘定系刷新とデジタルチャネル強化を柱とする全店DX計画の概要
----
タイトル: 営業店ペーパーレス化 全店展開レポート（2026年度上期）
match_count: 2
preview: ... パーセント 削減 した 紙 の 稟議 書 を 電子 化 する こと で 決裁 までの 日数 も 短縮 され た 今後 は 融資 部門 への 展開 を 進める DX 推進 室 は 効果 測定 を 継続 する


### 〈ここで覚えてほしいこと〉

検索結果には `title` / `match_count` / `preview` が含まれます。  
`match_count` は **検索キーワードが何回出てきたか** で、多く出るページほど上に並びます。  
シンプルですが、検索エンジンの基本的な考え方の一つです。

```
検索対象文字列を作る → 出現回数を数える → ヒットしたページを集める → match_count順に並べる → previewを付ける
```

これで **全文検索（`search_fulltext`）** ができました。  
次の **Step 4** で、検索の **読み込み元を DB に差し替え**、DB に貯めたデータを全文検索できるようにします。

---


---

## ✅ Step 3 まとめ — Jupyter で作った関数を `search_fulltext.py` に書き出そう

Step 3 では Jupyter 上で **2つの関数** を作りました。

| 関数 | 役割 |
|------|------|
| `search_fulltext(query, pages)` | 全文検索の本体（`match_count` 降順で返す） |
| `_make_preview(text, query)` | マッチ箇所の前後を切り出す補助関数 |

この2つを **`search_fulltext.py`** というファイルに書き出せば、`app.py` から呼び出せます。

```
Jupyter（このノートブック）              search_fulltext.py
──────────────────────────────   →   ──────────────────────────────
_make_preview() を理解した        →   同じ関数を書く（補助関数）
search_fulltext() を理解した      →   同じ関数を書く（本体）
                                                ↓
                                      app.py から
                                      from search_fulltext import search_fulltext
                                      で呼び出して使う
```

> 💡 **疎結合のポイント**：`search_fulltext(query, pages)` は **リストを受け取るだけ**。
> だから中身のデータが JSON 由来でも DB 由来でも動きます。次の Step 4 で効いてきます。

---

**いまの成果物の状態**

| ファイル | 状態 |
|----------|------|
| `app.py` | 🟡 W2のまま（次の Step 4 で差し替え） |
| `crawler.py` | ✅ W2完成版を流用 |
| `schema.sql` / `database.py` | ✅ 完成 |
| `search_fulltext.py` | ✅ **完成**（`match_count` 版・本文まで検索できる） |

**➡️ 次は Step 4**：最後の仕上げ。`app.py` の検索タブを **DBを読む** 形に差し替えます。


---
# Step 4：この検索を app.py に組み込む（DB → 全文検索の仕上げ）

> ### 🎯 〈このStepのねらい〉
>
> 部品はすべて揃いました。最後は **検索の「読み込み元」を JSON から DB に差し替えるだけ**。
> たった2行で済むこと——それ自体が、部品を疎結合に作ってきたご褒美です。
>
> **このStepが終わったときにできていること**
> - `get_all_pages()` → `search_fulltext()` の2行で検索が通ることを確認できる
> - `search_fulltext()` を **1行も変えずに** データ源だけ差し替えられた理由を説明できる
> - `app.py` の検索タブを DB ベースに書き換えられる
>
> **触るファイル**：`app.py`（検索タブの読み込み元を差し替え）

Step 3 で、**DB から読み込んだデータ**（`get_all_pages()`）を全文検索できました。
あとは、この2行を **アプリ（`app.py`）の検索タブ** に載せるだけです。

```python
# app.py の検索タブの中身（これだけ）
pages   = get_all_pages()               # DB から全ページを読む（Step 2）
results = search_fulltext(query, pages) # 全文検索する（Step 3）
```

> 💡 **疎結合のポイント**：`search_fulltext(query, pages)` は `pages`（リスト）を受け取るだけ。
> だから **データ源が JSON でも DB でも、中身を1行も変えずに** そのまま動きます。W3 では読み込み元を DB にしただけで、検索ロジックは無変更です。
> （※ 賢い並び替え＝TF-IDF は来週 W4）


In [16]:
# 【app.py と同じ2行で検索してみる】＝ DB → 全文検索の一本道の確認
# get_all_pages()（Step2で定義）と search_fulltext()（Step3で定義）をそのまま使う。
#   実ファイルの app.py では「from database import get_all_pages」と書くが、
#   このノートブックでは前セルで定義済みなので import は不要。

# 【差し替え】JSONではなくDBから pages を作る
pages = get_all_pages()
print(f"DBから読み込んだページ数：{len(pages)} 件")

# search_fulltext は Step 3 のものをそのまま使う（無変更）
query = "DX"
results = search_fulltext(query, pages)
print(f"'{query}' の検索結果：{len(results)} 件")
for r in results[:5]:
    print("----")
    print("タイトル:", r["title"])
    print("match_count:", r.get("match_count"))


DBから読み込んだページ数：36 件
'DX' の検索結果：7 件
----
タイトル: DX戦略レポート2024
match_count: 4
----
タイトル: 営業店DX推進レポート
match_count: 3
----
タイトル: AIチャットボットによる照会対応DX
match_count: 3
----
タイトル: 全店DX推進ロードマップ2026
match_count: 3
----
タイトル: 営業店ペーパーレス化 全店展開レポート（2026年度上期）
match_count: 2


---

## ✅ Step 4 まとめ

**このStepでやったこと**

| やったこと | ひとことで言うと |
|------------|------------------|
| 読み込み元をDBに差し替えた | `json.load()` → `get_all_pages()` |
| 検索ロジックは無変更で通した | `search_fulltext()` は1行も直さなくてよかった |
| 2行で検索できることを確認した | DB → 全文検索の一本道 |

**いまの成果物の状態**

| ファイル | 状態 |
|----------|------|
| `app.py` | ✅ **完成**（検索タブがDBを読んで全文検索する） |
| `crawler.py` | ✅ W2流用（保存先をDBに向ければ完了） |
| `schema.sql` / `database.py` | ✅ 完成 |
| `search_fulltext.py` | ✅ 完成（無変更で流用できた） |
| `data/tech0_search.db` | ✅ 検索対象データが入っている |

**➡️ 次は W3のふりかえり**：チェックリストで動作を確認し、W4 の課題（ランキング／TF-IDF）につなげます。


---
# ✅ W3まとめ ＋ 次週への引き継ぎ

> ✍️ W3 の完成コードは **`answers/w3/`**（app.py / database.py / schema.sql / search_fulltext.py / crawler.py）にあります。

### 今週やったこと
- `pages_w2.json`（ファイル）→ **SQLite（`data/tech0_search.db`）** にデータを移した（`migrate_from_json()`／2-3）
- `database.py`（`init_db` / `insert_page` / `get_all_pages`）で DB を読み書きできるようにした
- **全文検索（`search_fulltext`／match_count版）を作り**、本文まで検索できるようにした
- 検索の**読み込み先を DB に差し替え**た（データ源が JSON でも DB でも `search_fulltext` は共通）

### 〈DBの動作チェックリスト〉
- ☐ `data/tech0_search.db` が生成されている
- ☐ `get_all_pages()` で登録したページが取れる
- ☐ 同じURLを再登録しても重複せず**上書き**される（`INSERT OR REPLACE`）← **2-4 で確認済み**
- ☐ 検索が JSON ではなく **DB を読んでいる**

> 💬 **CDO黒崎（W4へ）**：「器は整った。だが並び順がまだ素朴だ。“出現回数が多い順”では、ありふれた語ばかり上位に来る。**Googleのように賢く並べろ**。」
> ➡️ 次週 **W4** で **TF-IDF** を導入し、`match_count` を賢いランキングに進化させます。DBにデータが揃った今なら、コーパス全体からIDFを計算できます。
